# 📂 Clase 1 — Lectura y escritura de archivos CSV
## Obtención de datos y fundamentos de Numpy

**Situación:** Eres parte del equipo de análisis comercial de una empresa distribuidora de productos de consumo masivo. Tu jefatura te envió el archivo `ventas_mensuales.csv` con el detalle de ventas del último mes.

Se espera que respondas:
- ¿Qué sucursales tienen mayor volumen de ventas?
- ¿Qué productos están destacando en cada ciudad?
- ¿Cómo preparar esta información para un informe de gerencia?

> ⚠️ El archivo contiene errores intencionales para practicar diagnóstico y limpieza.

---
## 📦 Librerías necesarias

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print('✅ Librerías cargadas correctamente')

---
## PASO 1 — Carga del archivo CSV

### 1.1 Lectura básica con `read_csv()`

In [ ]:
df = pd.read_csv('ventas_mensuales.csv')

print('=== Primeras filas del archivo ===')
print(df.head(10))

### 1.2 Diagnóstico inicial del DataFrame

In [ ]:
print('=== Información general ===')
print(df.info())
print()
print('=== Dimensiones del dataset ===')
print(f'Filas: {df.shape[0]} | Columnas: {df.shape[1]}')
print()
print('=== Nombres de columnas ===')
print(df.columns.tolist())

### ✏️ Ejercicio 1.2 — Responde:

In [ ]:
# ✏️ ¿Los nombres de columnas tienen sentido? ¿Hay alguna columna innecesaria?
respuesta_columnas = ""

# ✏️ ¿Qué tipos de datos detectaste en cada columna? ¿Hay algo inesperado?
respuesta_tipos = ""

print(respuesta_columnas)
print(respuesta_tipos)

---
## PASO 2 — Diagnóstico y limpieza de datos

### 2.1 Detectar valores faltantes

In [ ]:
print('=== Valores faltantes por columna ===')
print(df.isnull().sum())
print()
print('=== Filas con valores faltantes ===')
print(df[df.isnull().any(axis=1)])

### 2.2 Detectar inconsistencias en columnas de texto

In [ ]:
print('=== Valores únicos en columna sucursal ===')
print(df['sucursal'].unique())
print()
print('=== Valores únicos en columna categoria ===')
print(df['categoria'].unique())

### ✏️ Ejercicio 2.2 — Detecta los errores intencionales:

In [ ]:
# ✏️ Describe los errores que encontraste:
errores_detectados = """
Error 1 (valor faltante):       
Error 2 (valor no numérico):    
Error 3 (formato de fecha):     
Error 4 (inconsistencia texto): 
"""
print(errores_detectados)

### 2.3 Corrección de errores

In [ ]:
# Paso a) Convertir 'cantidad' a numérico (errors='coerce' transforma no-numéricos a NaN)
df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce')

# Paso b) Convertir 'precio' a numérico
df['precio'] = pd.to_numeric(df['precio'], errors='coerce')

# Paso c) Estandarizar nombres de sucursal (capitalizar correctamente)
df['sucursal'] = df['sucursal'].str.title()

# Paso d) Eliminar filas con NaN en columnas críticas
df_limpio = df.dropna(subset=['cantidad', 'precio'])

print(f'Filas originales:  {len(df)}')
print(f'Filas después de limpieza: {len(df_limpio)}')
print(f'Filas eliminadas:  {len(df) - len(df_limpio)}')
print()
print('=== Sucursales únicas (corregidas) ===')
print(df_limpio['sucursal'].unique())

### ✏️ Ejercicio 2.3 — Reflexión:

In [ ]:
# ✏️ ¿Por qué usamos errors='coerce' y no errors='raise' en pd.to_numeric()?
respuesta_coerce = ""

# ✏️ ¿Qué alternativa existe a eliminar las filas con NaN? ¿Cuándo usarías cada opción?
respuesta_alternativa = ""

print(respuesta_coerce)
print(respuesta_alternativa)

---
## PASO 3 — Cálculo y análisis

### 3.1 Calcular columna `total_venta`

In [ ]:
df_limpio = df_limpio.copy()
df_limpio['total_venta'] = df_limpio['cantidad'] * df_limpio['precio']

print('=== DataFrame con total_venta ===')
print(df_limpio[['fecha','sucursal','producto','cantidad','precio','total_venta']].head(10))

### 3.2 Identificar ventas destacadas (total_venta > $10.000)

In [ ]:
df_destacadas = df_limpio[df_limpio['total_venta'] > 10000]

print(f'Transacciones destacadas: {len(df_destacadas)} de {len(df_limpio)}')
print()
print('=== Productos más frecuentes en ventas destacadas ===')
print(df_destacadas['producto'].value_counts().head(8))

### ✏️ Ejercicio 3.2 — Analiza:

In [ ]:
# ✏️ ¿Son los productos destacados de alto precio o de alta cantidad?
# Investiga usando las columnas cantidad y precio:
print('=== Estadísticas de ventas destacadas ===')
print(df_destacadas[['cantidad','precio','total_venta']].describe())

# ✏️ Escribe tu conclusión:
conclusion_destacadas = ""
print('\nConclusión:', conclusion_destacadas)

---
## PASO 4 — Agrupación y exportación

### 4.1 Agrupar por sucursal

In [ ]:
ventas_por_sucursal = df_limpio.groupby('sucursal')['total_venta'].sum().sort_values(ascending=False)

print('=== Total de ventas por sucursal ===')
for sucursal, total in ventas_por_sucursal.items():
    print(f'  {sucursal:<20} ${total:>12,.0f}')

### 4.2 Agrupar por producto y categoría

In [ ]:
ventas_por_producto = (
    df_limpio.groupby(['producto','categoria'])
    .agg(unidades_vendidas=('cantidad','sum'), total_ventas=('total_venta','sum'))
    .sort_values('total_ventas', ascending=False)
    .reset_index()
)

print('=== Top 10 productos por ventas totales ===')
print(ventas_por_producto.head(10).to_string(index=False))

### 4.3 Visualización

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Análisis de Ventas Mensuales — Empresa Distribuidora', fontsize=13, fontweight='bold')

# Gráfico 1: Ventas por sucursal
colores = ['#1F4E79','#2E75B6','#4472C4','#70AD47','#ED7D31','#FFC000','#FF0000','#7030A0']
axes[0].barh(ventas_por_sucursal.index[::-1], ventas_por_sucursal.values[::-1], color=colores)
axes[0].set_title('Total Ventas por Sucursal ($)', fontweight='bold')
axes[0].set_xlabel('Total ($)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[0].grid(axis='x', linestyle='--', alpha=0.4)

# Gráfico 2: Top 5 productos
top5 = ventas_por_producto.head(5)
axes[1].bar(top5['producto'], top5['total_ventas'], color='#2E75B6', edgecolor='white')
axes[1].set_title('Top 5 Productos por Ventas ($)', fontweight='bold')
axes[1].set_ylabel('Total ($)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
axes[1].grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('ventas_grafico.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico guardado como ventas_grafico.png')

### 4.4 Exportar archivos CSV con `to_csv()`

In [ ]:
# Archivo 1: Resumen por sucursal (para gerencia)
ventas_por_sucursal.to_csv('ventas_resumen.csv', sep=';', header=True)
print('✅ ventas_resumen.csv generado')

# Archivo 2: Solo ventas destacadas
df_destacadas[['fecha','sucursal','producto','cantidad','precio','total_venta']].to_csv(
    'ventas_destacadas.csv', sep=';', index=False, float_format='%.0f'
)
print('✅ ventas_destacadas.csv generado')

# Archivo 3: Resumen completo para Power BI (sin índice, con punto y coma)
df_limpio.to_csv('ventas_completo_limpio.csv', sep=';', index=False, 
                  float_format='%.0f', encoding='utf-8')
print('✅ ventas_completo_limpio.csv generado')

### 4.5 Verificar los archivos generados

In [ ]:
print('=== Verificación: ventas_resumen.csv ===')
verificado = pd.read_csv('ventas_resumen.csv', sep=';')
print(verificado)
print()
print('=== Verificación: ventas_destacadas.csv (primeras 5 filas) ===')
verificado2 = pd.read_csv('ventas_destacadas.csv', sep=';')
print(verificado2.head())
print(f'Total filas: {len(verificado2)}')

---
## PASO 5 — Preguntas de reflexión

### ✏️ Responde las preguntas de cierre del ejercicio:

In [ ]:
# ✏️ 1. ¿Qué decisiones podría tomar la gerencia a partir del archivo ventas_resumen.csv?
decision_gerencia = ""

# ✏️ 2. ¿Qué limitaciones ves en este análisis si solo usamos CSV y no bases de datos?
limitaciones = ""

# ✏️ 3. ¿Cómo podrías automatizar este proceso para ejecutarlo cada mes?
automatizacion = ""

print('--- REFLEXIONES ---')
print(f'1. Decisiones de gerencia:  {decision_gerencia}')
print(f'2. Limitaciones del CSV:    {limitaciones}')
print(f'3. Automatización mensual:  {automatizacion}')

---
## PASO 6 — Desafío extra: leer un CSV desde URL

Pandas puede leer archivos CSV directamente desde internet. Practica con datos públicos:

In [ ]:
# ✏️ Desafío: lee el siguiente CSV público y muestra las primeras 5 filas
url = 'https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv'

# df_url = pd.read_csv(url)
# print(df_url.head())
# print(df_url.info())

# ✏️ ¿Cuántas filas y columnas tiene?
# ✏️ ¿Qué tipo de datos contiene?
print('Descomenta las líneas de arriba para ejecutar el desafío.')

---
## 📋 Resumen de parámetros clave

| Función | Parámetro | Uso |
|---------|-----------|-----|
| `read_csv()` | `sep=';'` | Separador punto y coma (Chile/Europa) |
| `read_csv()` | `encoding='utf-8'` | Tildes y ñ correctas |
| `read_csv()` | `header=None` | Archivo sin cabecera |
| `read_csv()` | `skiprows=2` | Saltar filas iniciales |
| `read_csv()` | `usecols=['col1','col2']` | Cargar solo columnas específicas |
| `to_csv()` | `index=False` | No escribir el índice |
| `to_csv()` | `sep=';'` | Separador punto y coma |
| `to_csv()` | `float_format='%.2f'` | Dos decimales |
| `to_csv()` | `date_format='%Y-%m-%d'` | Formato de fecha ISO |

> 💡 **Regla de oro:** Antes de procesar un CSV, ábrelo con un editor de texto para identificar visualmente el separador, la codificación y si tiene cabecera.